# Consolidating retriever (golden + labrador) health and desexing data

## Purpose
Prepare one traceable dataset from four published PDF tables covering male and female Labrador Retrievers and Golden Retrievers. The resulting CSV is intended for import into PostgreSQL for exploratory analysis.

**Research question:** Is age at desexing associated with the occurrence and timing of selected cancers and joint disorders, and how do these patterns differ by breed and sex?

## Approach to interpretation

This dataset accompanies the research paper cited below. I have not read any of their findings, I will defer a detailed review of the results and discussion until after completing my initial exploratory analysis. This is intended to reduce the influence of prior expectations while allowing me to develop and document my own observations.

I will consult the study methods and dataset key to understand the variables, inclusion criteria and limitations. After completing the analysis, I will compare my findings with the published results and discuss any differences in methods or interpretation.

## Sources
Hart et al. (2014), *Long-Term Health Effects of Neutering Dogs: Comparison of Labrador Retrievers with Golden Retrievers*.

- [Research paper](https://doi.org/10.1371/journal.pone.0102241)
- [Published dataset](https://doi.org/10.6084/m9.figshare.1038819)
- Local reference: `dataset/Dataset Key.pdf`

## Run instructions
Open this notebook in the project root beside `dataset/`, select a Python environment with `pandas` and `pdfplumber`, and run all cells in order. If needed, install them once with `pip install pandas pdfplumber`.

**Output:** `dataset/csv/dogs_combined.csv` (20 columns). Rerunning replaces this generated CSV.


## 1. Setup and column mapping
Map the published headings to SQL-friendly names. Retain the original disease abbreviations until interpretation is checked against the dataset key.

In [1]:
from pathlib import Path
import pdfplumber
import pandas as pd

header_map = {
    "Unique Identifier": "dog_id",
    "DOB": "dob",
    "Sex": "sex",
    "Neuter Status": "neuter_status",
    "Neuter Date": "neuter_date",
    "Intact Start Date": "intact_start_date",
    "Intact End Date": "intact_end_date",
    "Neuter Start Date": "neuter_start_date",
    "Neuter End Date": "neuter_end_date",
    "HD": "hd",
    "CCL": "ccl",
    "ED": "ed",
    "LSA": "lsa",
    "MCT": "mct",
    "HSA": "hsa",
    "MC": "mc",
    "PYO": "pyo"
}



## 2. Extract and consolidate
Check every page for expected headings and sex values, remove its header row, and stack the records. Source filename and page number allow each record to be traced back to its PDF.

In [2]:
frames = []

pdf_files = sorted(Path("dataset").glob("*Retriever Data*.pdf"))
assert len(pdf_files) == 4, "Expected exactly four retriever PDFs."

for pdf_path in pdf_files:
    breed = (
        "Labrador Retriever"
        if "Labrador" in pdf_path.name
        else "Golden Retriever"
    )
    expected_sex = "f" if pdf_path.name.startswith("Female") else "m"
    expected_columns = 17 if expected_sex == "f" else 15

    with pdfplumber.open(pdf_path) as pdf:
        for page_number, page in enumerate(pdf.pages, start=1):
            tables = page.extract_tables()

            if not tables:
                raise ValueError(
                    f"No table: {pdf_path.name}, page {page_number}"
                )

            for table in tables:
                headers = [
                    " ".join((cell or "").split())
                    for cell in table[0]
                ]

                if (
                    len(headers) != expected_columns
                    or len(set(headers)) != len(headers)
                    or set(headers) != set(list(header_map)[:expected_columns])
                ):
                    raise ValueError(
                        f"Unexpected headings: {pdf_path.name}, "
                        f"page {page_number}: {headers}"
                    )

                data = pd.DataFrame(
                    table[1:],
                    columns=[header_map[h] for h in headers]
                ).fillna("")

                if not data["sex"].str.strip().eq(expected_sex).all():
                    raise ValueError(
                        f"Check sex values: {pdf_path.name}, page {page_number}"
                    )

                data["breed"] = breed
                data["source_file"] = pdf_path.name
                data["source_page"] = page_number
                frames.append(data)

column_order = (
    ["dog_id", "breed"]
    + list(header_map.values())[1:]
    + ["source_file", "source_page"]
)

dogs = pd.concat(frames, ignore_index=True)
dogs = dogs.reindex(columns=column_order)



## 3. Validate the consolidated records
Check identifiers and reconcile counts with the initial extraction. These checks detect structural problems; they do not replace comparing extracted values with the PDFs.

In [3]:
if dogs["dog_id"].str.strip().eq("").any():
    raise ValueError("Blank dog identifiers found; inspect the source pages.")

if dogs.duplicated(["breed", "sex", "dog_id"]).any():
    raise ValueError("Repeated breed/sex/ID combinations found; inspect before exporting.")

summary = dogs.groupby(["breed", "sex"]).size().rename("records")
expected = {
    ("Golden Retriever", "f"): 471,
    ("Golden Retriever", "m"): 543,
    ("Labrador Retriever", "f"): 692,
    ("Labrador Retriever", "m"): 808,
}
if summary.to_dict() != expected:
    raise ValueError("Counts differ from the initial extraction; investigate before exporting.")

print(f"Validated {len(dogs):,} records across {len(summary)} groups.")
display(summary.reset_index())
display(dogs.head(5))

Validated 2,514 records across 4 groups.


,breed,sex,records
0,Golden Retriever,f,471
1,Golden Retriever,m,543
2,Labrador Retriever,f,692
3,Labrador Retriever,m,808


,dog_id,breed,dob,sex,neuter_status,neuter_date,intact_start_date,intact_end_date,neuter_start_date,neuter_end_date,hd,ccl,ed,lsa,mct,hsa,mc,pyo,source_file,source_page
0,GR001xx,Golden Retriever,12/12/2010,f,< 6 mo.,1/1/2011,,,4/5/2012,4/5/2012,,,,,,,,,Female Golden Retriever Data (Hart et al.).pdf,1
1,GR002xx,Golden Retriever,10/28/2003,f,< 6 mo.,11/27/2003,,,1/5/2005,1/13/2005,,,,,,,,,Female Golden Retriever Data (Hart et al.).pdf,1
2,GR003xx,Golden Retriever,4/29/2006,f,< 6 mo.,5/29/2006,,,5/1/2007,6/12/2007,,,,,,,,,Female Golden Retriever Data (Hart et al.).pdf,1
3,GR004xx,Golden Retriever,8/15/2004,f,< 6 mo.,9/15/2004,,,7/27/2011,7/27/2011,8/15/2005,,,,,,,,Female Golden Retriever Data (Hart et al.).pdf,1
4,GR005xx,Golden Retriever,7/1/2001,f,< 6 mo.,8/9/2001,,,5/28/2003,6/5/2003,,,,invalid,,,,,Female Golden Retriever Data (Hart et al.).pdf,1


## 4. Export for PostgreSQL
Write one CSV in a fixed column order. Empty fields remain empty in the CSV; PostgreSQL import settings determine whether they become NULL.

In [4]:
output_path = Path("dataset/csv/dogs_combined.csv")
output_path.parent.mkdir(parents=True, exist_ok=True)
dogs.to_csv(output_path, index=False)

# Verify that exporting and reading back preserves field values and column order.
round_trip = pd.read_csv(output_path, dtype=str, keep_default_na=False)
pd.testing.assert_frame_equal(
    round_trip, dogs.fillna("").astype(str), check_dtype=False
)
print(f"Exported {len(dogs):,} records and {len(dogs.columns)} columns to {output_path}")

Exported 2,514 records and 20 columns to dataset/csv/dogs_combined.csv


## Next stage: SQL exploration
Import the CSV into `lab_desex.public.dogs_raw` using the same column order, then reconcile counts by breed and sex.